# 🎙️ Train Wake Words: "Hey Money Penny" & "Hey Sheila"

This notebook trains two custom wake word models using [openWakeWord](https://github.com/dscripka/openWakeWord).

**What it does:**
1. Installs dependencies (Piper TTS, openWakeWord, etc.)
2. Downloads training datasets (RIRs, background audio, negative features)
3. Generates synthetic speech clips for each wake phrase
4. Augments clips with noise/reverb for robustness
5. Trains two models and exports as `.onnx` + `.tflite`

**Runtime:** ~60-90 min total on a T4 GPU (free Colab tier works fine)

**Output:** Two model files in Google Drive:
- `hey_money_penny.onnx`
- `hey_sheila.onnx`

---
⚡ **Instructions:** Runtime → Change runtime type → T4 GPU, then Run All (Ctrl+F9)

## 1. Environment Setup

In [ ]:
# Install piper-sample-generator (TTS for synthetic training data)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install -q piper-phonemize webrtcvad

# Install openWakeWord with training support
!git clone https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword

# Training dependencies
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
    speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
    acoustics==0.2.6 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19

# TFLite conversion deps
!pip install -q tensorflow-cpu==2.8.1 tensorflow_probability==0.16.0 onnx_tf==1.10.0

# Download embedding/melspec models (needed by openWakeWord)
import os
os.makedirs('./openwakeword/openwakeword/resources/models', exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx \
    -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite \
    -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx \
    -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite \
    -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

print('✅ Environment setup complete!')

In [ ]:
# Imports
import os
import sys
import numpy as np
import yaml
import datasets
import scipy
from pathlib import Path
from tqdm import tqdm

## 2. Download Training Datasets

This downloads:
- **MIT Room Impulse Responses** — for realistic reverb augmentation
- **AudioSet + Free Music Archive** — background noise/music
- **ACAV100M features** (16GB) — pre-computed negative examples (2,000 hrs of audio)
- **Validation features** — for false-positive rate estimation

⏱️ Takes ~10-15 min

In [ ]:
# Room Impulse Responses (MIT)
output_dir = './mit_rirs'
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
for row in tqdm(rir_dataset, desc='Downloading RIRs'):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print(f'✅ {len(os.listdir(output_dir))} RIR files')

In [ ]:
# Background audio: AudioSet
os.makedirs('audioset', exist_ok=True)
!wget -q -O audioset/bal_train09.tar \
    'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar'

# If AudioSet download fails (404), skip it — FMA alone is sufficient
audioset_ok = os.path.getsize('audioset/bal_train09.tar') > 1000
if audioset_ok:
    !cd audioset && tar -xf bal_train09.tar
    output_dir = './audioset_16k'
    os.makedirs(output_dir, exist_ok=True)
    audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
    audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset, desc='Converting AudioSet'):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    print(f'✅ AudioSet: {len(os.listdir(output_dir))} files')
else:
    print('⚠️ AudioSet download failed — will use FMA only')

# Background audio: Free Music Archive
output_dir = './fma'
os.makedirs(output_dir, exist_ok=True)
fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))

n_hours = 2  # 2 hours of background music
for i in tqdm(range(n_hours * 3600 // 30), desc='Downloading FMA'):
    try:
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    except StopIteration:
        break
print(f'✅ FMA: {len(os.listdir(output_dir))} files')

In [ ]:
# Pre-computed negative features (16GB — Colab has ~100GB disk, no problem)
!wget -q --show-progress \
    https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# Validation features for false-positive rate
!wget -q --show-progress \
    https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

print('✅ Feature files downloaded')
!ls -lh *.npy

## 3. Configure & Train Models

We train two models sequentially:
1. **hey_money_penny** — activates Moneypenny (Dustin's assistant)
2. **hey_sheila** — activates Sheila (Jess's assistant)

Each model takes ~20-30 min to train on a T4 GPU.

### Training parameters (production quality)
- 50,000 positive + negative synthetic samples each
- 5,000 validation samples
- 50,000 training steps
- Target: accuracy ≥ 0.7, recall ≥ 0.5, FP rate ≤ 0.2/hr

In [ ]:
# Build background_paths based on what we downloaded
background_paths = ['./fma']
if os.path.exists('./audioset_16k') and len(os.listdir('./audioset_16k')) > 0:
    background_paths.insert(0, './audioset_16k')

# ═══════════════════════════════════════════════════
# CONFIGURE BOTH WAKE WORDS HERE
# ═══════════════════════════════════════════════════

WAKE_WORDS = [
    {
        'target_phrase': ['hey money penny'],
        'model_name': 'hey_money_penny',
        'output_dir': './model_hey_money_penny',
        # Phrases that sound similar but should NOT trigger activation
        'custom_negative_phrases': [
            'hey monopoly', 'hey many penny', 'a money penny',
            'pay money penny', 'hey honey penny', 'hey money bunny',
            'hey money plenty', 'hey money twenty'
        ],
    },
    {
        'target_phrase': ['hey sheila'],
        'model_name': 'hey_sheila',
        'output_dir': './model_hey_sheila',
        'custom_negative_phrases': [
            'hey sheela', 'hey shelly', 'hey shelia',
            'say sheila', 'hey sealer', 'hey shield',
            'hey she la', 'hey dealer'
        ],
    },
]

print(f'Will train {len(WAKE_WORDS)} models: {[w["model_name"] for w in WAKE_WORDS]}')

In [ ]:
# Generate YAML configs for both models

config_template = yaml.load(
    open('openwakeword/examples/custom_model.yml', 'r').read(), yaml.Loader
)

for ww in WAKE_WORDS:
    config = config_template.copy()
    config['target_phrase'] = ww['target_phrase']
    config['model_name'] = ww['model_name']
    config['output_dir'] = ww['output_dir']
    config['custom_negative_phrases'] = ww.get('custom_negative_phrases', [])

    # Production-quality training params
    config['n_samples'] = 50000
    config['n_samples_val'] = 5000
    config['steps'] = 50000
    config['target_accuracy'] = 0.7
    config['target_recall'] = 0.5
    config['target_false_positives_per_hour'] = 0.2
    config['max_negative_weight'] = 1500

    # Data paths
    config['background_paths'] = background_paths
    config['background_paths_duplication_rate'] = [1] * len(background_paths)
    config['false_positive_validation_data_path'] = './validation_set_features.npy'
    config['feature_data_files'] = {
        'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    }
    config['batch_n_per_class'] = {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50
    }
    config['piper_sample_generator_path'] = './piper-sample-generator'

    # Save config
    config_path = f'{ww["model_name"]}.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f)
    print(f'✅ Saved config: {config_path}')
    print(f'   Phrase: {ww["target_phrase"]}')
    print(f'   Samples: {config["n_samples"]} train / {config["n_samples_val"]} val')
    print(f'   Steps: {config["steps"]}')
    print()

### Train Model 1: `hey_money_penny`

Three steps: generate clips → augment → train

In [ ]:
# Step 1: Generate synthetic clips for "hey money penny"
# ~15-20 min on T4 GPU for 50k samples
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_money_penny.yaml --generate_clips

In [ ]:
# Step 2: Augment clips with noise + reverb
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_money_penny.yaml --augment_clips

In [ ]:
# Step 3: Train the model
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_money_penny.yaml --train_model
print('\n✅ hey_money_penny training complete!')

### Train Model 2: `hey_sheila`

In [ ]:
# Step 1: Generate synthetic clips for "hey sheila"
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_sheila.yaml --generate_clips

In [ ]:
# Step 2: Augment clips
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_sheila.yaml --augment_clips

In [ ]:
# Step 3: Train
!{sys.executable} openwakeword/openwakeword/train.py --training_config hey_sheila.yaml --train_model
print('\n✅ hey_sheila training complete!')

## 4. Export Models to Google Drive

Save the trained `.onnx` models to your Drive for easy download.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
output_drive = '/content/drive/MyDrive/wake_word_models'
os.makedirs(output_drive, exist_ok=True)

models_found = []
for ww in WAKE_WORDS:
    name = ww['model_name']
    out_dir = ww['output_dir']
    for ext in ['.onnx', '.tflite']:
        src = os.path.join(out_dir, f'{name}{ext}')
        if os.path.exists(src):
            dst = os.path.join(output_drive, f'{name}{ext}')
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1024**2
            print(f'✅ Saved: {dst} ({size_mb:.1f} MB)')
            models_found.append(dst)
        else:
            print(f'⚠️ Not found: {src}')

# Also save the configs for reference
for ww in WAKE_WORDS:
    shutil.copy2(f'{ww["model_name"]}.yaml', os.path.join(output_drive, f'{ww["model_name"]}.yaml'))

print(f'\n🎉 Done! {len(models_found)} model files saved to Google Drive: {output_drive}')

## 5. Quick Test (Optional)

Load the models and verify they work with openWakeWord.

In [ ]:
import openwakeword
from openwakeword.model import Model

# Test each model loads correctly
for ww in WAKE_WORDS:
    model_path = os.path.join(ww['output_dir'], f'{ww["model_name"]}.onnx')
    if os.path.exists(model_path):
        owwModel = Model(wakeword_models=[model_path])
        # Feed some silence to verify inference works
        silence = np.zeros(1280, dtype=np.int16)
        for _ in range(5):
            prediction = owwModel.predict(silence)
        print(f'✅ {ww["model_name"]}: loaded OK, inference working')
        print(f'   Prediction on silence: {prediction}')
    else:
        print(f'❌ {ww["model_name"]}: model file not found at {model_path}')

print('\n🎙️ Models are ready! Download from Google Drive → wake_word_models/')